# Compare the current SciPy and Pyomo shelf-temperature optimizers

This tutorial revisits the historical $A_1 \times K_C$ shelf-temperature experiment using the implementation on current `main`. It intentionally does **not** recreate the removed Pyomo.DAE collocation model. Instead, it asks the comparable current question: when both workflows use the SciPy completion time as a common horizon, how do the current sequential SciPy optimizer and simultaneous backward-Euler Pyomo optimizer compare?

## What changed from the historical experiment?

| Historical experiment | Current-main experiment |
| --- | --- |
| Free final-time Pyomo.DAE problem | Fixed-horizon Pyomo trajectory problem |
| Finite differences and orthogonal collocation | Backward Euler on an explicit uniform grid |
| Minimize drying time | Minimize `sum(Pch - Psub)` subject to a final drying target |
| Compare final-time objectives directly | Compare an integrated driving-force objective on a shared horizon |

The shared horizon avoids presenting unlike objectives as though they were identical. The SciPy workflow still runs until complete drying; its completion time becomes the Pyomo horizon, and Pyomo must reach the configured final dried fraction within that time.

## Optional solver setup

Run this notebook from the repository root after installing the optional stack:

```bash
python -m pip install -e ".[dev,pyomo]"
idaes get-extensions --extra petsc
jupyter lab docs/examples/current_main_optimizer_comparison.ipynb
```

A conda-forge IPOPT installation can be used instead. The full default grid takes several minutes because every SciPy reference advances with a 0.01-hour step and timing is repeated.

In [ ]:
# Papermill parameters: CI overrides these with a small smoke case.
a1_values = [16.0, 18.0, 20.0]
kc_values = [2.75e-4, 3.30e-4, 4.00e-4]
scipy_dt = 0.01
n_steps = 24
final_dried_fraction = 0.989
timing_repeats = 3
mesh_steps = [12, 24, 48]
constraint_tolerance = 1.0e-4
save_results = False
results_dir = "benchmarks/results/current_main_tsh_comparison"

In [ ]:
import json
import platform
import subprocess
from importlib.metadata import version
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pyomo
import scipy

from examples.current_main_optimizer_comparison import (
    run_case_comparison,
    run_mesh_sensitivity,
    run_pyomo_at_horizon,
)

In [ ]:
try:
    git_revision = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    git_revision = "unavailable"

environment = {
    "git_revision": git_revision,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "lyopronto": version("lyopronto"),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "pyomo": pyomo.__version__,
    "matplotlib": matplotlib.__version__,
}
environment

## Experimental protocol

For each $(A_1, K_C)$ pair, the helper performs paired runs:

1. Run `opt_Tsh.dry` to 100% dried using the legacy 0.01-hour sequential step.
2. Use that completion time as the horizon of the current Pyomo shelf-temperature model.
3. Require Pyomo to reach 98.9% dried, matching the nonsingular target convention.
4. Integrate $P_{ch}-P_{sub}$ in Torr over time for both seven-column trajectories.
5. Repeat the paired timing measurement and report medians.

The reported speedup is therefore a **workflow-level** comparison: roughly 1,200 sequential SciPy points versus a simultaneous Pyomo model with `n_steps + 1` nodes. It is not a per-iteration solver benchmark.

In [ ]:
# Pay one-time Pyomo/IPOPT initialization before collecting timings.
warmup = run_pyomo_at_horizon(
    a1_values[0],
    kc_values[0],
    horizon_hr=2.0,
    n_steps=4,
    final_dried_fraction=0.10,
)
assert warmup.success, (warmup.solver_status, warmup.termination_condition)

In [ ]:
comparisons = []
for a1 in a1_values:
    for kc in kc_values:
        print(f"Running A1={a1:g}, KC={kc:.2e} ...")
        comparisons.append(
            run_case_comparison(
                a1,
                kc,
                scipy_dt=scipy_dt,
                n_steps=n_steps,
                final_dried_fraction=final_dried_fraction,
                timing_repeats=timing_repeats,
            )
        )

case_by_parameter = {(case.a1, case.kc): case for case in comparisons}
print(f"Completed {len(comparisons)} grid cases.")

In [ ]:
header = (
    f"{'A1':>5} {'KC':>10} {'horizon [h]':>12} {'objective gap [%]':>18} "
    f"{'SciPy [s]':>11} {'Pyomo [s]':>11} {'speedup':>10} {'final dried [%]':>16}"
)
print(header)
print("-" * len(header))
for case in comparisons:
    print(
        f"{case.a1:5.1f} {case.kc:10.2e} {case.horizon_hr:12.4f} "
        f"{case.objective_gap_percent:18.3f} {case.scipy_wall_median_s:11.3f} "
        f"{case.pyomo_wall_median_s:11.3f} {case.speedup:10.1f} "
        f"{case.pyomo_trajectory[-1, 6]:16.3f}"
    )

## Validate before interpreting

A solver's `optimal` termination is not sufficient by itself. The experiment also checks the final drying target, finite seven-column outputs, the product-temperature limit, and current Pyomo constraint residuals. Legacy output pressure remains in mTorr and percent dried remains on a 0–100 scale.

In [ ]:
for case in comparisons:
    assert case.scipy_trajectory.shape[1] == 7
    assert case.pyomo_trajectory.shape == (n_steps + 1, 7)
    assert np.all(np.isfinite(case.scipy_trajectory))
    assert np.all(np.isfinite(case.pyomo_trajectory))
    assert case.pyomo_trajectory[-1, 6] >= 100.0 * final_dried_fraction - 1.0e-3
    assert np.max(case.pyomo_trajectory[:, 2]) <= -25.0 + 1.0e-4
    assert case.max_constraint_violation <= constraint_tolerance

print("All grid cases satisfy the experiment acceptance checks.")

In [ ]:
def matrix_for(attribute):
    return np.array(
        [
            [getattr(case_by_parameter[(float(a1), float(kc))], attribute) for kc in kc_values]
            for a1 in a1_values
        ],
        dtype=float,
    )

def annotate_heatmap(ax, values, fmt):
    threshold = 0.55 * np.nanmax(np.abs(values))
    for row in range(values.shape[0]):
        for col in range(values.shape[1]):
            color = "white" if abs(values[row, col]) > threshold else "black"
            ax.text(col, row, format(values[row, col], fmt), ha="center", va="center", color=color)

objective_gaps = matrix_for("objective_gap_percent")
limit = max(1.0, float(np.max(np.abs(objective_gaps))))
fig, ax = plt.subplots(figsize=(7.5, 4.5))
image = ax.imshow(objective_gaps, cmap="coolwarm", vmin=-limit, vmax=limit, aspect="auto")
ax.set(
    title="Current Pyomo integrated driving-force gap from SciPy (%)",
    xlabel="ht.KC [cal/s/K/cm²]",
    ylabel="product.A1 [cm·hr·Torr/g]",
)
ax.set_xticks(range(len(kc_values)), [f"{value:.2e}" for value in kc_values], rotation=35)
ax.set_yticks(range(len(a1_values)), [f"{value:g}" for value in a1_values])
annotate_heatmap(ax, objective_gaps, ".2f")
fig.colorbar(image, ax=ax, label="relative objective gap [%]")
fig.tight_layout()
plt.show()

In [ ]:
speedups = matrix_for("speedup")
fig, ax = plt.subplots(figsize=(7.5, 4.5))
image = ax.imshow(speedups, cmap="viridis", aspect="auto")
ax.set(
    title=f"Current Pyomo workflow speedup over SciPy (median of {timing_repeats})",
    xlabel="ht.KC [cal/s/K/cm²]",
    ylabel="product.A1 [cm·hr·Torr/g]",
)
ax.set_xticks(range(len(kc_values)), [f"{value:.2e}" for value in kc_values], rotation=35)
ax.set_yticks(range(len(a1_values)), [f"{value:g}" for value in a1_values])
annotate_heatmap(ax, speedups, ".1f")
fig.colorbar(image, ax=ax, label="SciPy median / Pyomo median")
fig.tight_layout()
plt.show()

The speedup values are machine- and environment-dependent. They should be interpreted together with the discretization sizes and timing repeats, not used as regression-test thresholds. The objective gaps and physical acceptance checks are the more portable numerical results.

## Backward-Euler mesh sensitivity

Current `main` intentionally has one trajectory discretization. Instead of reproducing obsolete FD-versus-collocation panels, we check whether its integrated objective moves toward the dense SciPy reference as the number of backward-Euler steps increases.

In [ ]:
nominal = case_by_parameter[(float(a1_values[0]), float(kc_values[0]))]
mesh_rows = run_mesh_sensitivity(
    nominal.a1,
    nominal.kc,
    nominal.scipy_trajectory,
    n_steps_values=mesh_steps,
    final_dried_fraction=final_dried_fraction,
)

print(f"{'steps':>7} {'objective gap [%]':>18} {'final dried [%]':>17} {'max residual':>14}")
for row in mesh_rows:
    print(
        f"{int(row['n_steps']):7d} {row['objective_gap_percent']:18.3f} "
        f"{row['final_percent_dried']:17.3f} {row['max_constraint_violation']:14.3e}"
    )

fig, ax = plt.subplots(figsize=(6.5, 4.0))
ax.plot(
    [row["n_steps"] for row in mesh_rows],
    [row["objective_gap_percent"] for row in mesh_rows],
    marker="o",
)
ax.axhline(0.0, color="black", linewidth=1.0, linestyle="--", label="SciPy reference")
ax.set(xlabel="Backward-Euler steps", ylabel="objective gap [%]", title="Nominal-case mesh sensitivity")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## Nominal trajectory comparison

Both results retain the legacy seven-column output shape. Pressure is plotted from column 4 in mTorr and drying progress from column 6 on the 0–100 percent scale. Pyomo ends at 98.9% by design; SciPy advances to 100%.

In [ ]:
scipy_table = nominal.scipy_trajectory
pyomo_table = nominal.pyomo_trajectory
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)

for table, label, style in [
    (scipy_table, "SciPy sequential", {"linewidth": 2.0}),
    (pyomo_table, "Current Pyomo backward Euler", {"linestyle": "--", "marker": "o"}),
]:
    axes[0, 0].plot(table[:, 0], table[:, 3], label=label, **style)
    axes[0, 1].plot(table[:, 0], table[:, 1], label=label, **style)
    axes[1, 0].plot(table[:, 0], table[:, 4], label=label, **style)
    axes[1, 1].plot(table[:, 0], table[:, 6], label=label, **style)

axes[0, 0].set(title="Shelf temperature", ylabel="temperature [°C]")
axes[0, 1].set(title="Sublimation-front temperature", ylabel="temperature [°C]")
axes[0, 1].axhline(-25.0, color="tab:red", linestyle=":", label="product limit")
axes[1, 0].set(title="Chamber pressure", xlabel="time [h]", ylabel="pressure [mTorr]")
axes[1, 1].set(title="Drying progress", xlabel="time [h]", ylabel="dried [%]")
axes[1, 1].axhline(100.0 * final_dried_fraction, color="tab:red", linestyle=":", label="Pyomo target")
for ax in axes.flat:
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle(f"Current-main comparison: A1={nominal.a1:g}, KC={nominal.kc:.2e}", fontweight="bold")
fig.tight_layout()
plt.show()

## Optionally save a compact reproduction record

Generated experiment outputs belong under `benchmarks/results/`, which is ignored by default. The JSON file contains environment and scalar summaries; the compressed NumPy archive retains every trajectory used for the plots.

In [ ]:
if save_results:
    destination = Path(results_dir)
    destination.mkdir(parents=True, exist_ok=True)
    summaries = [
        {
            "A1": case.a1,
            "KC": case.kc,
            "horizon_hr": case.horizon_hr,
            "scipy_objective": case.scipy_objective,
            "pyomo_objective": case.pyomo_objective,
            "objective_gap_percent": case.objective_gap_percent,
            "scipy_wall_times_s": list(case.scipy_wall_times_s),
            "pyomo_wall_times_s": list(case.pyomo_wall_times_s),
            "speedup": case.speedup,
            "final_percent_dried": float(case.pyomo_trajectory[-1, 6]),
            "max_constraint_violation": case.max_constraint_violation,
        }
        for case in comparisons
    ]
    (destination / "summary.json").write_text(
        json.dumps({"environment": environment, "parameters": {
            "scipy_dt": scipy_dt, "n_steps": n_steps,
            "final_dried_fraction": final_dried_fraction,
            "timing_repeats": timing_repeats,
        }, "cases": summaries, "mesh_sensitivity": mesh_rows}, indent=2) + "\n"
    )
    trajectories = {}
    for case in comparisons:
        key = f"A1_{case.a1:g}_KC_{case.kc:.2e}".replace(".", "p").replace("-", "m")
        trajectories[f"{key}_scipy"] = case.scipy_trajectory
        trajectories[f"{key}_pyomo"] = case.pyomo_trajectory
    np.savez_compressed(destination / "trajectories.npz", **trajectories)
    print(f"Saved reproduction record under {destination}")
else:
    print("Set save_results=True to write JSON and NPZ outputs.")

## Interpretation boundaries

This notebook reproduces the **scientific comparison on current `main`**, not the historical implementation. Agreement should be judged using the integrated objective, mesh trend, trajectory shape, drying target, temperature limit, and constraint residuals together. Exact speedup annotations are not portable across machines, and the Pyomo APIs remain optional validation prototypes rather than drop-in replacements for the legacy SciPy optimizers.